<a href="https://colab.research.google.com/github/gibthom12-arch/Wumpus_World_AI/blob/main/AiCourseWorkImplementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Note for proper running please start this cell first
# (Note the values in this cell correspond to test case 6 if you wish to study the expected outputs)
# If you wish to use test cases please remember to also run the reset state cell below this atleast once

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
import IPython.display
from IPython.display import clear_output
from IPython.display import display
from ipywidgets import Output

out = Output()
display(out)


# Agent Reasoning Variables Storing its understanding of the Environment

safe = [] # Guaranteed safe tiles
FullPath = [] # The full path the agent takes to travel to each target position. (Permits repeated coordinates)
Travelled = [] # The list of all target positions the agent has travelled. (Does not permit repeated coordinates)
Uncertain = [] # The list of all possibly dangerous tiles with a value holding their likelihood of danger
ProcessedSense = [] # Senses Tiles already processed by the agent
PossibleWumpus = [] # Areas where the wumpus could be along with a value of the likelihood of its presence
PossibleSniper = [] # Areas where the sniper could be along with value of the likelihood of its presence
BulletPath = [] # The path the bullet travels after a sniper shoots

# Map Creation variables holding information to form the Environment
Gridsize = 6
move = True
Start_Pos = (0, 0)
robot_pos = Start_Pos
NumEp = 0

PIT = 1
BREEZE = 2
WUMPUS = 3
SMELLY = 4
BREEZE_SMELL = 5
GOLD = 7
ARROW = 8
SNIPER = 9
PEW = 10
Sniper = [(0, 5)]

# Booleans for aspects of the environment which may dynamically change during runtime
GoldFound = False
ArrowUsed = False
ArrowFound = False
AgentAlive = True
WumpusAlive = True
SniperAlive = True
Reloading = False

# Forms a the layout of the grid
def Form_Grid(pit_pos = (4, 3), gold_pos = (5, 5), wumpus_pos = (3, 4), arrow_pos = (1, 1)):
    em_grid = np.zeros((Gridsize, Gridsize), dtype = int)
    Pit = [(pit_pos)]
    Gold = [(gold_pos)]
    Wumpus = [(wumpus_pos)]
    Wx, Wy = Wumpus[0]
    Smelly = FindAdjacent(Wx, Wy)
    x2, y2 = Pit[0]
    Breeze = FindAdjacent(x2, y2)
    Arrow = [(arrow_pos)]
    Sx,  Sy = Sniper[0]
    Pew = FindAdjacent(Sx, Sy)
    INTERACTIBLES = [(Pit, PIT), (Breeze, BREEZE), (Gold, GOLD), (Wumpus, WUMPUS), (Smelly, SMELLY), (Arrow, ARROW), (Sniper, SNIPER), (Pew, PEW)]

    for position, value in INTERACTIBLES:
        for x, y in position:
            if x < Gridsize and y < Gridsize:
                if em_grid[x, y] == 0:
                    if (WumpusAlive == False and (value == SMELLY or value == WUMPUS)) or (SniperAlive == False and (value == SNIPER or value == PEW) or (value == ARROW and ArrowUsed == True)):
                        em_grid[x, y] = 0
                    else:
                        em_grid[x, y] = value

                elif em_grid[x, y] == BREEZE and value == SMELLY:
                    if WumpusAlive == True:
                        em_grid[x, y] = BREEZE_SMELL
                    else:
                        em_grid[x, y] = BREEZE

                elif em_grid[x, y] == SMELLY and value == BREEZE:
                    if WumpusAlive == True:
                        em_grid[x, y] = BREEZE_SMELL
                    else:
                        em_grid[x, y] = BREEZE

    return em_grid

# Displays grid based on layout defined by form_grid()
def Show_Grid(grid, robot_pos):
    x, y = robot_pos

    fig, ax = plt.subplots()
    display_grid = np.copy(grid)
    display_grid[x, y] = 6
    ax.imshow(display_grid, vmin = -2, vmax = 11)

    ax.set_xlim(-0.5, Gridsize - 0.5)
    ax.set_ylim(-0.5, Gridsize - 0.5)
    ax.set_xticks(np.arange(-0.5, Gridsize, 1), minor = True)
    ax.set_yticks(np.arange(-0.5, Gridsize, 1), minor = True)
    ax.grid(which='minor')
    ax.set_xticks(range(Gridsize))
    ax.set_yticks(range(Gridsize))

    for i in range(Gridsize):
        for j in range(Gridsize):
            val = grid[i, j]
            text = ""
            if val == PIT:
                text = "X"
            if val == BREEZE:
                text = "B"
            if val == GOLD:
                text = "G"
            if val == WUMPUS:
                text = "W"
            if val == SMELLY:
                text = "S"
            if val == BREEZE_SMELL:
                text = "B/S"
            if val == ARROW:
                text = "A"
            if val == SNIPER:
              text = "SNIPER"
            if val == PEW:
              text = "PEW"
            if val == 6:
                text = "R"
            ax.text(j, i, text, ha='center', va='center', fontsize=12)

    with out:
      clear_output(wait=True)
      display(fig)
    plt.close(fig)

# Checks if the agent is alive
def Check_Dead(robot_pos, grid):
    x, y = robot_pos
    global AgentAlive

    if grid[x, y] == PIT or grid[x, y] == WUMPUS or grid[x, y] == SNIPER or (x, y) in BulletPath:
        AgentAlive = False

# Finds the adjacent tiles to an entered coordinate
def FindAdjacent(x, y):
    Adjacent = []
    for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
        nx, ny = x + dx, y + dy
        if Gridsize > nx >= 0 and Gridsize > ny >= 0:
            Adjacent.append((nx, ny))

    return Adjacent

# Checks if there are any moves left in safe that are untravelled
def Safe_Moves_Left():
    for nodes in safe:
        if nodes not in Travelled:
            return True
    return False

# Main reasoning function where the agent determines its next action
def Reasoning(grid, robot_pos):
    x, y = robot_pos
    Adjacent = FindAdjacent(x, y)
    NoSafeLeft = False
    SmellFound = False
    actions = None

    # Ensures the agents current tile is added to safe
    if robot_pos not in safe:
        safe.append((x, y))

    # Adds agents current tile to Travelled
    Travelled_Update(x, y)

    if grid[x, y] == ARROW:
        global ArrowFound
        global ArrowUsed
        ArrowFound = True
        ArrowUsed = True
        print("Arrow Found")

    if grid[x, y] == GOLD:
        actions = Start_Pos
        global GoldFound
        GoldFound = True
        return actions

    # Implements rule that if a tile has no dangerous sense all adjacent tiles must be safe
    if grid[x, y] != BREEZE and grid[x, y] != SMELLY and grid[x, y] != BREEZE_SMELL and grid[x, y] != PEW:
        for i in range(0, len(Adjacent)):
            if Adjacent[i] not in safe:
                safe.append(Adjacent[i])
                print("No senses at: ", robot_pos, " Adding valid adjacent tiles marked safe")
        actions = Agent_Control(grid, Adjacent)
        print("Descision made moving to ", actions)

    # If dangerous tile present activates
    elif grid[x, y] == BREEZE or grid[x, y] == SMELLY or grid[x, y] == BREEZE_SMELL:
        if actions != None:
          print("Danger in adjacent tiles to ", robot_pos)
        Adjacent = FindAdjacent(x, y)
        z, t = x, y
        SmellFound = False

        # Used to determine if agent near wumpus
        if grid[x, y] == SMELLY or grid[x, y] == BREEZE_SMELL:
            SmellFound = True
            print("Agent near wumpus")

        # Adds tiles to Uncertain if the tile isnt already in safe
        for Node in Adjacent:
            if Node not in safe:
                tempUncertain = []

                # Note the code contained in smellfound == true is purely to find and kill the wumpus. The code below is used to decide movement
                # Code runs for only smell tiles
                if SmellFound == True:
                    tempPossibleWumpus = []
                    #Checks if each adjacent node to the smell has been added to the list of possible wumpus tile
                    for i in range (0, len(PossibleWumpus)):
                        x, y, val = PossibleWumpus[i]
                        tempPossibleWumpus.append((x, y))

                    # If it is a new tile not in possible wumpus it is added and given a base risk value of 1
                    if Node not in tempPossibleWumpus:
                        x, y = Node
                        NewVal = x, y, 1
                        PossibleWumpus.append(NewVal)

                    # If the adjacent nodes already been noted as a risky square in PossibleWumpus
                    # If it has and current smell node has not been processed the adjacent nodes risk value is increased by 1 in PossibleWumpus
                    elif (z, t) not in ProcessedSense:
                        for i in range(0, len(PossibleWumpus)):
                            x, y, val = PossibleWumpus[i]
                            if Node == (x, y):
                                val += 1
                                PossibleWumpus[i] = x, y, val


                # Code runs for both smell and breeze tiles
                # Checks if adjacent nodes are already in uncertain
                for i in range(0, len(Uncertain)):
                    x, y, val = Uncertain[i]
                    tempUncertain.append((x, y))

                # If adjacent nodes not already in uncertain adds to uncertain with value of
                if Node not in tempUncertain:
                    x, y = Node
                    NewVal = x, y, 1
                    Uncertain.append(NewVal)
                    print(Node, " added to uncertain")

                # If adjacent nodes in uncertain and the current sense not already processed
                # risk value increased by 1
                elif (z, t) not in ProcessedSense:
                    ProcessedSense.append((z, t))
                    for i in range(0, len(Uncertain)):
                        x, y, val = Uncertain[i]
                        if Node == (x, y):
                            val += 1
                            Uncertain[i] = x, y, val
                            print(Node, " risk value has been increased to ", val)

        # Checks if requirements are met to kill wumpus
        if SmellFound == True and ArrowFound == True and len(PossibleWumpus) > 0:
            bestSus = 0
            SusLocation = (0, 0)
            # Takes the tile with the highest chance of having wumpus
            for i in range(0, len(PossibleWumpus)):
                x, y, val = PossibleWumpus[i]
                if val > bestSus:
                    bestSus = val
                    SusLocation = (x, y)
            # Requires evidence of atleast 2 to ensure right answer is picked
            if bestSus > 2:
                print("Wumpus deduced location at ", SusLocation, "with suspicion score of", bestSus)
                Shoot_Wumpus(SusLocation, grid)

        i = len(FullPath) - 1
        j = len(Travelled) - 1
        deciding = False

        while deciding == False and len(FullPath) > 0:
            i -= 1
            if i < 0:
              actions = robot_pos
              deciding = True
              break
            actions = FullPath[i]
            if actions != (z, t) and actions != Travelled[j]:
                deciding = True
            else:
              print("Possibly get stuck")

    # Used when all safe tiles are travelled to and the agent is not at the start position (This is as at beginning agent starts with nothing in safe)
    if Safe_Moves_Left() == False and robot_pos != (0, 0):
        Lowestrisk = 1000
        LowestUncertainty = 0, 0
        count = 0

        if len(Uncertain) == 0:
            return

        # Travels to node with the lowest risk value from uncertain
        for i in range(0, len(Uncertain)):
            x, y, value = Uncertain[i]

            if value < Lowestrisk:
                Lowestrisk = value
                LowestUncertainty = (x, y)
                count = i

        actions = LowestUncertainty
        Uncertain.remove(Uncertain[count])
        print("No safe moves left - moving towards ", actions, "because it has the lowest risk")

    # For encountering the sniper obstacle
    if grid[x, y] == PEW and SniperAlive == True:
        # Checks all adjacent nodes to its current location and finds the best choice which must not be in bullet path
        print("Sniper is found agent moving outside range")
        for nodes in Adjacent:
            if nodes not in BulletPath:
                if nodes in safe:
                    actions = nodes
                if actions == None:
                    actions = nodes

            # To determine snipers position and find it
            if nodes not in safe:
                # Adds unseen adjacent tiles to PossibleSniper with a base risk of 1
                if nodes not in PossibleSniper:
                    Sx, Sy = nodes
                    PossibleSniper.append((Sx, Sy, 1))

                # Otherwise appends the value of it in possible sniper
                else:
                    for i in range(0, len(PossibleSniper)):
                        x, y, val = PossibleSniper[i]
                        if nodes == (x, y):
                            val += 1
                            PossibleSniper[i] = x, y, val

        # Checks if requirements to find sniper are met
        for nodes in PossibleSniper:
            x, y, val = nodes
            """ Uses value of 0 as the sniper only has a maximum of 2 sense nodes for the agent to collect information from
                And there is no downside to choosing incorrectly """
            if val > 0:
                print("Sniper position deduced at ", (x, y), "with suspicion score of", val)
                Find_Sniper((x, y), grid)
    return actions

# Code to find and defeat sniper if requirements are met in reasoning
def Find_Sniper(SnipeTile, grid):
    x, y = SnipeTile
    if grid[x, y] == SNIPER:
        global BulletPath
        global SniperAlive
        SniperAlive = False
        print("Sniper killed well done, it was shot at ", SnipeTile)
        PossibleSniper.clear()
        BulletPath.clear()

        # Adds the sniper to safe tiles now that it is defeated
        if (x, y) not in safe:
            safe.append((x, y))


# Kills wumpus if the requirements are met
def Shoot_Wumpus(ShotTile, grid):
    global ArrowFound
    x, y = ShotTile

    # If inputted coordinates are correct the wumpus is killed
    if grid[x, y] == WUMPUS:
        global WumpusAlive
        WumpusAlive = False
        print("Wumpus killed well done, it was shot at ", ShotTile)
        PossibleWumpus.clear()

        # Adds wumpus old position to safe if not already included
        if (x, y) not in safe:
            safe.append((x, y))

        # Removes values associated to wumpus in uncertain
        for nodes in Uncertain:
            dx, dy, value = nodes
            if (dx, dy) == (x, y):
                Uncertain.remove(nodes)
    else:
        print("Missed at", ShotTile)

    ArrowFound = False

# Defines how the sniper mechanic works
def Sniper_Shoot(grid, SniperPos, robot_pos):
    global Reloading
    global BulletPath
    x, y = robot_pos

    # Checks to ensure the agent is on a pew tile and the sniper is not reloading
    if grid[x, y] == PEW and Reloading == False:
        direction = 0
        Sx, Sy = SniperPos
        # The sniper fires a shot covering the entire row or column of tiles associating to the specific pew tile the agent is on
        if x == Sx:
            if y > Sy:
              direction = 1
              for i in range(Sy, Gridsize):
                  BulletPath.append((Sx, i))
            else:
              direction = -1
              for i in range(Sy, -1, -1):
                  BulletPath.append((Sx, i))

        elif y == Sy:
            if x > Sx:
              direction = 1
              for i in range(Sx, Gridsize):
                  BulletPath.append((i, Sy))
            else:
              direction = -1
              for i in range(Sx, -1, -1):
                  BulletPath.append((i, Sy))
        # Once the sniper ativates it begins reloading
        Reloading = True
        print("Sniper fired across", BulletPath)

    # If the sniper has already been in reloading and is called again at the next loop its reloading finishes
    elif Reloading == True:
        BulletPath = []
        Reloading = False
        print("Sniper reloaded")

# Checks the distance between two points using mahattan distance
def Distance(Start, End):
    distance = abs(End[0] - Start[0]) + abs(End[1] - Start[1])
    return distance

# Adds values to Travelled
def Travelled_Update(x, y):
    if (x, y) not in Travelled:
        Travelled.append((x, y))

# Moves the agent to a target position from where it currently is
def Move_To_Target(grid, robotpos, targetpos):
    x, y = robotpos
    tx, ty = targetpos
    path = []
    TargetAdjacent = FindAdjacent(tx, ty)
    steps = 0
    reached = False

    if robotpos == targetpos:
        print("stuck")
        actions = None
        return actions

    if len(FullPath) == 0:
        FullPath.append(robotpos)

    # Checks if agent already is adjacent or at its target position in which case it can just move directly to the target
    if robotpos in TargetAdjacent or robotpos == targetpos:
        Travelled_Update(tx, ty)
        i = len(FullPath)

        if targetpos != FullPath[i - 1]:
            FullPath.append(targetpos)
        return targetpos

    # Loops until the agent is adjacent to its target
    while (x, y) not in TargetAdjacent:
        ValidMoves = [] # List of all valid moves the agent can make
        values = []
        current = (x, y)
        Adjacent = FindAdjacent(x, y)
        best_distance = 10000
        best_move = 0,0

        # Adds Travelled nodes to valid moves if they are adjacent to the agent and not already in its path
        for node in Travelled:
            if node in Adjacent and node not in path:
                ValidMoves.append(node)

        if not ValidMoves:
            print("No valid moves")
            break

        # Calculates which move in valid move has the shortest distance to the target position
        for moves in ValidMoves:
            distance = Distance(moves, targetpos)

            if distance < best_distance:
                best_distance = distance
                best_move = moves

        next_step = best_move
        x, y = next_step

        if next_step == current:
            print("Stuck")
            break

        path.append(next_step)

    # Checks to ensure the agent is adjacent to the target and updates reached to true
    if (x, y) in TargetAdjacent:
        reached = True

    # Displays the path the agent takes to get to the target
    for nodes in path:
        i = len(FullPath)
        if FullPath[i - 1] != nodes:
            FullPath.append(nodes)
        Show_Grid(grid, nodes)

    # If it successfully reaches target the target positon added to Travelled
    if reached == True:
        Travelled_Update(tx, ty)
        return targetpos
    else:
        return (x, y)

# Finds safe tiles that are not travelled for the agent to move towards
def Agent_Control(grid, Adjacent):
    for i in range(0, len(safe)):
        if safe[i] not in Travelled:
            actions = safe[i]
            break
        else:
            actions = None
    return actions

# Makes the initial grid
grid = Form_Grid(pit_pos = (4, 3), gold_pos = (5, 5), wumpus_pos = (3, 4), arrow_pos = (1, 1))

# Ensures agent is alive at the beginnning of the program
Check_Dead(robot_pos, grid)

while NumEp < 60 and AgentAlive == True:
    # If the sniper is alive it activates the sniper to check if agent is on a pew tile
    if SniperAlive == True:
      Sniper_Shoot(grid, Sniper[0], robot_pos)

    # Updates map if sniper is killed
    elif SniperAlive ==  False:
      grid = Form_Grid(pit_pos = (4, 3), gold_pos = (5, 5), wumpus_pos = (3, 4), arrow_pos = (1, 1))

    # Updates map if wumpus is killed
    if WumpusAlive == False:
        grid = Form_Grid(pit_pos = (4, 3), gold_pos = (5, 5), wumpus_pos = (3, 4), arrow_pos = (1, 1))

    # Updates map if arrow is found
    if ArrowUsed == True:
        print("Arrow Found")
        grid = Form_Grid(pit_pos = (4, 3), gold_pos = (5, 5), wumpus_pos = (3, 4), arrow_pos = (1, 1))

    # Prints if gold is found
    if GoldFound == True:
        print("Gold Found")
        break

    # Gets the action to undertake from reasoning
    actions = Reasoning(grid, robot_pos)

    # Check to ensure there is an outputted action for the agent to make
    if actions is None:
        print("No safe moves")
        break

    # Passes the action along with the robots position and the grid into move to target to move the agent to its target
    robot_pos = Move_To_Target(grid, robot_pos, actions)
    Show_Grid(grid, robot_pos) # Updates the map after the agent moves its position
    Check_Dead(robot_pos, grid) # Checks if the agent is still alive after moving its positon
    NumEp = NumEp + 1

if WumpusAlive == False:
    print("Wumpus died in:", NumEp, "Episodes")
if GoldFound == True and robot_pos == Start_Pos:
  print("Agent passed gold was found in: ", NumEp, " Episodes")
elif AgentAlive == False:
  print("Agent died in:", NumEp, "Episodes")
else:
  print("Agent run out of episodes")

plt.show()


Output()

No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 0)
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 1)
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 0)
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 1)
Arrow Found
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 2)
Arrow Found
No senses at:  (0, 2)  Adding valid adjacent tiles marked safe
Descision made moving to  (3, 0)
Arrow Found
No senses at:  (3, 0)  Adding valid adjacent tiles marked safe
No senses at:  (3, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 1)
Arrow Found
No senses at:  (2, 1)  Addin

In [1]:
# Resets the state of the map to its original state
def Reset_State(pit_pos, gold_pos, wumpus_pos, arrow_pos, sniper_pos):
  global safe, FullPath, Travelled, Uncertain, ProcessedSense
  global PossibleWumpus, PossibleSniper, BulletPath
  global robot_pos, NumEp
  global WumpusAlive, SniperAlive, AgentAlive, GoldFound, ArrowFound, ArrowUsed, Reloading
  global Sniper, grid

  safe.clear()
  FullPath.clear()
  Travelled.clear()
  Uncertain.clear()
  ProcessedSense.clear()
  PossibleWumpus.clear()
  PossibleSniper.clear()
  BulletPath.clear()

  robot_pos = (0, 0)
  NumEp = 0
  GoldFound = False
  ArrowUsed = False
  ArrowFound = False
  AgentAlive = True
  WumpusAlive = True
  SniperAlive = True
  Reloading = False
  Sniper = [(sniper_pos)]

  grid = Form_Grid(pit_pos, gold_pos, wumpus_pos, arrow_pos)
  print("Arrow tile value at", arrow_pos, "=", grid[arrow_pos[0], arrow_pos[1]])

# Implements the values inputted for the test cases
def Run_Test(TestName, pit_pos, gold_pos, wumpus_pos, arrow_pos, sniper_pos):
  global robot_pos, NumEp, AgentAlive, GoldFound, grid

  print("Test: ", TestName)
  print("Pit location: ", pit_pos, " | Wumpus location: ", wumpus_pos, " | Sniper location: ", sniper_pos)
  print("Gold Location: ", gold_pos, "Arrow Location: ", arrow_pos)

  Reset_State(pit_pos, gold_pos, wumpus_pos, arrow_pos, sniper_pos)

  while NumEp < 60 and AgentAlive == True:
    # If the sniper is alive it activates the sniper to check if agent is on a pew tile
    if SniperAlive == True:
      Sniper_Shoot(grid, Sniper[0], robot_pos)

    # Updates map if sniper is killed
    elif SniperAlive ==  False:
      grid = Form_Grid(pit_pos, gold_pos, wumpus_pos, arrow_pos)

    # Updates map if wumpus is killed
    if WumpusAlive == False:
        grid = Form_Grid(pit_pos, gold_pos, wumpus_pos, arrow_pos)

    # Updates map if arrow is found
    if ArrowUsed == True:
        grid = Form_Grid(pit_pos, gold_pos, wumpus_pos, arrow_pos)

    # Prints if gold is found
    if GoldFound == True:
        print("Gold Found")
        break

    # Gets the action to undertake from reasoning
    actions = Reasoning(grid, robot_pos)

    # Check to ensure there is an outputted action for the agent to make
    if actions is None:
        print("No safe moves")
        break

    # Passes the action along with the robots position and the grid into move to target to move the agent to its target
    robot_pos = Move_To_Target(grid, robot_pos, actions)
    Show_Grid(grid, robot_pos) # Updates the map after the agent moves its position
    Check_Dead(robot_pos, grid) # Checks if the agent is still alive after moving its positon
    NumEp += 1

  if WumpusAlive == False:
    print("Wumpus died in:", NumEp, "Episodes")
  else:
    print("Wumpus survived")
  if GoldFound == True and robot_pos == Start_Pos:
    print("Agent passed gold was found in: ", NumEp, " Episodes")
  elif AgentAlive == False:
    print("Agent died in:", NumEp, "Episodes")
  else:
    print("Agent run out of episodes")



In [ ]:
Run_Test(TestName = "Test 1", pit_pos = (5, 5), gold_pos = (2, 2), wumpus_pos = (0, 5), arrow_pos = (1, 1), sniper_pos = (5, 0))
""" Standard Navigation - Hazards are isolated and not directly blocking the gold
 Display adgents navigation and ability to traverse the map

- Agent starts at (0, 0) and detects no danger sense and marks adjacent tiles as safe and continues its journey
- Agent collects arrow at (1,1)
- Agent encounters PEW tile (4, 0) and shoots down the path (5, 0), (4, 0), (3, 0), (2, 0), (1, 0), (0, 0)
- Agent then deduces snipers position is at (5, 0) and kills the sniper
- Agent continues exploration till it finds gold
- Agent does not kill wumpus as it finds the gold prior to collecting enough information to deduce its position
- Expected result Agent passed and gold was found in 14 episodes """


Test:  Test 1
Pit location:  (5, 5)  | Wumpus location:  (0, 5)  | Sniper location:  (5, 0)
Gold Location:  (2, 2) Arrow Location:  (1, 1)
Arrow tile value at (1, 1) = 8
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 0)
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 1)
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 0)
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 1)
Arrow Found
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 2)
No senses at:  (0, 2)  Adding valid adjacent tiles marked safe
Descision made moving to  (3, 0)
No senses at:  (3, 0)  Adding valid adjacent tiles ma

' Standard Navigation - Hazards are isolated and not directly blocking the gold\n Display adgents navigation and ability to traverse the map\n\n- Agent starts at (0, 0) and detects no danger sense and marks adjacent tiles as safe and continues its journey\n- Agent collects arrow at (1,1)\n- Agent encounters PEW tile (4, 0) and shoots down the path (5, 0), (4, 0), (3, 0), (2, 0), (1, 0), (0, 0)\n- Agent then deduces snipers position is at (5, 0) and kills the sniper\n- Agent continues exploration till it finds gold\n- Agent does not kill wumpus as it finds the gold prior to collecting enough information to deduce its position\n- Expected result Agent passed and gold was found in 14 episodes '

In [ ]:
Run_Test(TestName = "Test 2", pit_pos = (3, 3), gold_pos = (5, 5), wumpus_pos = (2, 2), arrow_pos = (3, 1), sniper_pos = (5, 0))
""" Central hazard with agent reasoning around multiple centralised threats
- Agent starts at (0, 0) and does not detect any dangerous senses and begins to travel
- Agent encounters smelly tile at (2, 1) and (2, 2) is added uncertain
- Agent encounters another smelly tile at (1, 2) and increases suspicion at (2, 2) to 2 and tile (1, 3) also added to uncertain
- Agent then encounters sniper at (4, 0) which fires along column 0
- Agent adds (5, 0) to possible sniper
- Agent moves out of the way and deduces sniper at (5, 0)
- Arrow found at (3, 1)
- Agent encounters smelly tile at (3, 2) increases (2, 2) suspicion risk value to 3 and adds (3, 3) to uncertain
- Wumpus position is deduced at (2, 2) and is killed
- Agent eventually finds gold
- Expected Result: passed with gold found and wumpus killed after 43 episodes
"""

Test:  Test 2
Pit location:  (3, 3)  | Wumpus location:  (2, 2)  | Sniper location:  (5, 0)
Gold Location:  (5, 5) Arrow Location:  (3, 1)
Arrow tile value at (3, 1) = 8
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 0)
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 1)
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 0)
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 1)
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 2)
No senses at:  (0, 2)  Adding valid adjacent tiles marked safe
Descision made moving to  (3, 0)
No senses at:  (3, 0)  Adding valid adjacent tiles marked safe
No

' Central hazard with agent reasoning around multiple centralised threats \n- Agent starts at (0, 0) and does not detect any dangerous senses and begins to travel\n- Agent encounters smelly tile at (2, 1) and (2, 2) is added uncertain\n- Agent encounters another smelly tile at (1, 2) and increases suspicion at (2, 2) to 2 and tile (1, 3) also added to uncertain\n- Agent then encounters sniper at (4, 0) which fires along column 0 \n- Agent adds (5, 0) to possible sniper\n- Agent moves out of the way and deduces sniper at (5, 0)\n'

In [ ]:
Run_Test(TestName = "Test 3", pit_pos = (2, 0), gold_pos = (4, 4), wumpus_pos = (5, 5), arrow_pos = (3, 3), sniper_pos = (0, 5))
""" Testing with agent extremely close to pit and the gold being directly adjacent to a pit
- Agent starts at (0, 0) adding adjacent tiles to safe as it has no dangerous sense
- After moving to (1, 0) it encounters a breeze and adds (2, 0) and (1, 1) to uncertain
- After some exploration chooses to travel to (2, 1) encountering another dangerous sense adding (3, 1) and (2, 2) to uncertain and increasing (2, 0) risk value to 2
- Continues exploration eventually encountering a PEW tile at (0, 4)
- The sniper then begins to fire down the the row of tiles [(0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 0)]
- The agent moves out of the way of the bullet path
- The agent then deduces the position of the sniper at (0, 5) with a suspicion score of 1 and kills it
- After some exploration agent moves to (3, 3) and finds arrow
- Then later on agent travels to (3, 0) encountering a dangerous tile increasing (2, 0) risk value to 3
- Agent then chooses to travel to (4, 4) finding the gold and choosing to immediately return to the starting position
- Expected output: Passes and Gold found after 30 episodes
- Although wumpus was not killed as agent found gold prior to collecting enough evidence

Test:  Test 3
Pit location:  (2, 0)  | Wumpus location:  (5, 5)  | Sniper location:  (0, 5)
Gold Location:  (4, 4) Arrow Location:  (3, 3)
Arrow tile value at (3, 3) = 8
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 0)
(2, 0)  added too uncertain
(1, 1)  added too uncertain
Descision made moving to  (0, 1)
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 1)
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 2)
No senses at:  (0, 2)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 1)
(3, 1)  added too uncertain
(2, 2)  added too uncertain
(2, 0)  risk value has been increased too  2
Descision made moving to  (1, 2)
No senses at:  (1, 2)  Adding vali

In [ ]:
Run_Test(TestName="Test 4", pit_pos=(5,5), gold_pos=(4,4), wumpus_pos=(4, 3), arrow_pos=(1,1), sniper_pos=(0,5))
""" Gold  placed close to the edge of the map and surrounded by hazards
- Agents starts at  (0, 0) beginning its journey and adding adjacent tiles to safe
- After some travelling the agent moves to (1, 1) finding the arrow
- It continues travelling not encountering any hazards for some time
- Eventualy encountering a PEW tile at (0, 4)
- The sniper begins to fire down the row of tiles [(0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 0)]
- The agent moves out of the way of the bullet path
- The snipers posiiton was then deduced at (5, 0) with a risk score of 1
- It continues travelling until it travels to (4, 2) encountering a smelly tile and adding (4, 3) to uncetain
- Agent chooses to backtrack and move to (3, 3) here it also encounters another smelly note increasing (4, 3) risk value to 2
- Then choosing to continue travelling safe nodes it knows of
- Eventually finding the gold and subsequently returning to the start position
- Expected Result: Passed and gold was found after 32 episodes
- Although wumpus survived as agent found gold prior to accumulating enough evidence """

Test:  Test 4
Pit location:  (5, 5)  | Wumpus location:  (4, 3)  | Sniper location:  (0, 5)
Gold Location:  (4, 4) Arrow Location:  (1, 1)
Arrow tile value at (1, 1) = 8
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 0)
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 1)
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 0)
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 1)
Arrow Found
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 2)
No senses at:  (0, 2)  Adding valid adjacent tiles marked safe
Descision made moving to  (3, 0)
No senses at:  (3, 0)  Adding valid adjacent tiles ma

' Gold  placed close to the edge of the map and surrounded by hazards\n- Agents starts at  (0, 0) beginning its journey and adding adjacent tiles to safe\n- After some travelling the agent moves to (1, 1) finding the arrow\n- It continues travelling not encountering any hazards for some time\n- Eventualy encountering a PEW tile at (0, 4)\n- The sniper begins to fire down the row of tiles [(0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 0)]\n- The agent moves out of the way of the bullet path\n- The snipers posiiton was then deduced at (5, 0) with a risk score of 1\n- It continues travelling until it travels to (4, 2) encountering a smelly tile and adding (4, 3) to uncetain\n- Agent chooses to backtrack and move to (3, 3) here it also encounters another smelly note increasing (4, 3) risk value to 2\n- Then choosing to continue travelling safe nodes it knows of\n- Eventually finding the gold and subsequently returning to the start position\n- Expected Result: Passed and gold was found after 

In [ ]:
Run_Test(TestName="Test 5", pit_pos=(2,2), gold_pos=(5,4), wumpus_pos=(4,5), arrow_pos=(3,0), sniper_pos=(0,5))
""" This test case is a combination of previous tests using a centralized hazard with a sniper and wumpus guarding either edge and gold being adjacent to the wumpus
- Agent starts at (0, 0) adding adjacent tiles to safe
- Agent travels without encountering anything until it reaches (3, 0) finding an arrow
- Agent eventually travels to (2, 1) which has a dangerous sense adding (2, 2) to uncertain
- Then choosing to backtrack and move to (1, 2) which also has a dangerous sense increasing (2, 2) risk value to 2 and adding (1, 3) to uncertain
- Agent travels without encountering danger until it reach (0, 4) and encounters a PEW tile
- The sniper begins to fire down the row of tiles [(0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 0)]
- The agent moves out of the way of the bullet path
- Sniper is deduced to be at position (0, 5) with suspicion of 1 and is killed
- After some time of travelling the agent eventually travels to (3, 2) which contains a dangerous sense increasing (2, 2) risk value to 3 and adding (3, 3) to uncertain
- It then chooses to backtrack and move to (2, 3) at which it increases the risk value of (3, 3) to 2
- After some travelling agent eventually encounters smelly tile at (4, 4) adding (5, 4) and (4, 5) to uncertain
- Agent chooses to move to (3, 5) it finds another smelly tile increasing (4, 5) risk value to 2
- It then chooses to continue travelling known safe nodes till it encounters the gold at (5, 4)
- It then immediately begins returning to the start position
- Expected result: Passed and gold found
- Although wumpus survived as agent found gold prior to accumulating enough evidence """


Test:  Test 5
Pit location:  (2, 2)  | Wumpus location:  (4, 5)  | Sniper location:  (0, 5)
Gold Location:  (5, 4) Arrow Location:  (3, 0)
Arrow tile value at (3, 0) = 8
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 0)
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 1)
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 0)
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 1)
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 2)
No senses at:  (0, 2)  Adding valid adjacent tiles marked safe
Descision made moving to  (3, 0)
Arrow Found
No senses at:  (3, 0)  Adding valid adjacent tiles ma

In [ ]:
Run_Test(TestName="Test 6", pit_pos = (4, 3), gold_pos = (5, 5), wumpus_pos = (3, 4), arrow_pos = (1, 1), sniper_pos = (0, 5))
""" This test is the same as the default one created for the main cell.
This test creates a wall if hazards guarding the gold forcing the agent to confront multiple risky nodes
- Agent starts at (0, 0) adding adjacent tiles to safe
- Agent traverses the map without finding anything until it travels to (1, 1) finding an arrow
- Following this after some time the agent travels to (0, 4) containing a PEW tile
- The sniper prepares to fire along the column [(0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 0)]
- The agent moves out of the bullet path
- The agent then deduces the snipers position at (0, 5) and kills the sniper
- After some travelling the agent chooses to move to (2, 4) at which it encounters a smelly tile at which (3, 4) and (2, 5) are added to uncertain
- Soon after this the agent travels to (4, 2) encountering a dangerous sense it adds (4, 3) to uncertain
- Then after travelling the (3, 3) and encountering the breeze smell it subsequently increases risk at (4, 3) to 2
- The agent eventually travels to (3, 5) adding (4, 5) to uncertain and increasing (3, 4) risk value to 2
- The agent then travels once more increasing (3, 4) risk value to 3 and subsequently deducing the wumpus position and killing it
- Following this the agent travels to (5, 3) encountering dangerous nodes and adding (5, 4) to uncertain while increasing (4, 3) risk value to 3 and then to 4
- The agent then later encounters a sense tile at (4, 4) increasing (5, 4) risk value to 2
- The agent runs out of safe moves to backtrack to following this so it picks (2, 5) as it has the lowest risk
- The agent once again lack any safe moves so it picks the tile with the next lowest risk (4, 5)
- It then moves to (5, 5) finding the gold and returning to start position
- Expected result: Passed and gold found after 42 episodes
- Wumpus is killed


Test:  Test 6
Pit location:  (4, 3)  | Wumpus location:  (3, 4)  | Sniper location:  (0, 5)
Gold Location:  (5, 5) Arrow Location:  (1, 1)
Arrow tile value at (1, 1) = 8
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
No senses at:  (0, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 0)
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
No senses at:  (1, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 1)
No senses at:  (0, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (2, 0)
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
No senses at:  (2, 0)  Adding valid adjacent tiles marked safe
Descision made moving to  (1, 1)
Arrow Found
No senses at:  (1, 1)  Adding valid adjacent tiles marked safe
Descision made moving to  (0, 2)
No senses at:  (0, 2)  Adding valid adjacent tiles marked safe
Descision made moving to  (3, 0)
No senses at:  (3, 0)  Adding valid adjacent tiles ma